In [1]:
#import useful stuff                                                                                                                                                                                                             
import numpy as np
import matplotlib.pyplot as plt
import os
import h5py
#check number of cores                                                                                                                                                                                                           
num_cores = os.cpu_count()
print(f"Number of CPU cores: {num_cores}")

N = 10000

Number of CPU cores: 96


In [2]:

ttbar_dist = np.load("/oscar/data/mleblan6/lhay/ttbar_distmatrix/full_distmatrix_ttbar.npy", mmap_mode='r')
print(np.shape(ttbar_dist))
# original_dist2neg = np.load("/oscar/data/mleblan6/SPECTER_hardprocess_dist2neg.npy", mmap_mode='r')
# print(np.shape(original_dist2neg))
jeppe_file = h5py.File("/oscar/data/mleblan6/cell_resampling/jeppe_100k_matrix_clustered.h5")
print(jeppe_file)
jeppe_dist_matrix = jeppe_file['distance_matrix'][:N, :N]
print(np.shape(jeppe_dist_matrix))

full_dist_matrix = jeppe_dist_matrix.copy()

iu = np.triu_indices_from(full_dist_matrix, k=1)

full_dist_matrix[iu[1], iu[0]] = full_dist_matrix[iu]


(100000, 100000)
<HDF5 file "jeppe_100k_matrix_clustered.h5" (mode r)>
(10000, 10000)


In [3]:
print(np.shape(jeppe_dist_matrix))
event_weights = np.load('/oscar/data/mleblan6/rjain/ppzjj_100k/weight_100k.npy')[:N]
neg_events = np.where(event_weights < 0)[0]
print(jeppe_dist_matrix==0)
print(np.sum(jeppe_dist_matrix==0)/np.sum(jeppe_dist_matrix>=0))

# for k in range(100):
#     row = jeppe_dist_matrix[k]
#     n_zeros = np.sum(row==0)
#     print(n_zeros)
    # order = np.argsort(row)
    # print("closest indices:", order[:10])
    # print("closest distances:", row[order[:10]])
    # print("closest weights:", event_weights[order[:10]])
    # print()

(10000, 10000)
[[ True False False ... False False False]
 [ True  True False ... False False False]
 [ True  True  True ... False False False]
 ...
 [False False False ...  True False False]
 [False False False ...  True  True False]
 [False False False ...  True  True  True]]
0.25005


In [4]:
def sort_dist(orig_dist_matrix, neg_events, N):
    sorted_dist_matrix = np.zeros((len(neg_events), N))
    for i in range(len(neg_events)):
        sorted_dist_matrix[i] = np.argsort(orig_dist_matrix[i])
        if i % 2000 == 0:
            print(i)
    
    return sorted_dist_matrix

In [5]:
#### in rishabhs version originalClose2neg = sorted_dist_matrix and original_dist2neg = orig_dist_matrix
def cell_reweight(max_radius, event_weights, dist_matrix, verbose = False):
    neg_events = np.where(event_weights < 0)[0]
    N = len(event_weights)
    reweighted_event_weight = np.copy(np.array(event_weights))
    cell_radius = []
    sorted_dist_matrix = sort_dist(dist_matrix, neg_events, N)
    if sorted_dist_matrix.shape[1] != N:
        raise ValueError(
            f"dist_matrix has {sorted_dist_matrix.shape[1]} columns, "
            f"but event_weights has length {N}"
        )

    if sorted_dist_matrix.shape[0] != len(neg_events):
        raise ValueError(
            f"dist_matrix has {sorted_dist_matrix.shape[0]} rows, "
            f"but neg_events has length {len(neg_events)}. "
            "This function assumes row i corresponds to neg_events[i]."
        )
        
    for neg_event_idx, event_idx in enumerate(neg_events):
        cell_weight = reweighted_event_weight[event_idx]
        abs_cell_weight = abs(reweighted_event_weight[event_idx])
        events_in_cell = [event_idx]
        final_cell_event = None
        if verbose and neg_event_idx % 2000 == 0:
            print(neg_event_idx)

        if reweighted_event_weight[event_idx] >= 0:
            #### this means the event was already rw'd in a different cell
            continue
        
        for j in range(N):
            close2neg = int(sorted_dist_matrix[neg_event_idx, j])
            if close2neg == event_idx:
                continue
            distance = dist_matrix[neg_event_idx, close2neg]
            if (cell_weight <= 0.01) and (distance < max_radius):
                neighbor_weight = reweighted_event_weight[close2neg]
                cell_weight += neighbor_weight
                abs_cell_weight += abs(neighbor_weight)
                events_in_cell.append(close2neg)
                final_cell_event = close2neg

            else:
                break


        if cell_weight > 0 and final_cell_event is not None:
            cell_radius = np.append(
                cell_radius,
                dist_matrix[neg_event_idx, final_cell_event]
            )
            reweighted_event_weight[events_in_cell] = (
                cell_weight / abs_cell_weight *
                abs(reweighted_event_weight[events_in_cell])
            )

        if verbose == True:
            if i % 2000 == 0:
                print(i)
    
    #print('Hard process reweighting completed')
    frac_rw = 1- (len(np.where(reweighted_event_weight < 0)[0]) / len(neg_events))
    print(frac_rw*100,f'% of negative weights are reweighted for hard process events at {max_radius} GeV cell radius')

    return reweighted_event_weight


In [6]:
max_rw = cell_reweight(1000, event_weights,jeppe_dist_matrix)


0
2000
99.94632313472893 % of negative weights are reweighted for hard process events at 1000 GeV cell radius


In [ ]:
radii = np.logspace(1,3,50)
reweights = []

for i, r in enumerate(radii):
    reweights.append(cell_reweight(r, event_weights, full_dist_matrix))

0
2000
0.0 % of negative weights are reweighted for hard process events at 10.0 GeV cell radius
0
2000
0.0 % of negative weights are reweighted for hard process events at 10.985411419875582 GeV cell radius
0
2000
0.0 % of negative weights are reweighted for hard process events at 12.067926406393289 GeV cell radius
0
2000
0.0 % of negative weights are reweighted for hard process events at 13.257113655901088 GeV cell radius
0
2000
0.0 % of negative weights are reweighted for hard process events at 14.563484775012435 GeV cell radius
0
2000
0.0 % of negative weights are reweighted for hard process events at 15.998587196060582 GeV cell radius
0
2000
0.0 % of negative weights are reweighted for hard process events at 17.57510624854792 GeV cell radius
0
2000
0.026838432635534204 % of negative weights are reweighted for hard process events at 19.306977288832496 GeV cell radius
0
2000
0.05367686527106841 % of negative weights are reweighted for hard process events at 21.209508879201902 GeV cell

In [ ]:
fracs = np.zeros(len(radii))

for i in range(len(radii)):
    fracs[i] = 1 - len(np.where(reweights[i] < 0)[0]) / len(neg_events)

In [ ]:
plt.plot(radii, fracs, marker = 'o', color = 'blue', linestyle = '--')
plt.axhline(y=1, color = 'black')
plt.xlabel('Max Cell Radius [GeV]')
plt.ylabel('Reweighted Fraction')
#plt.yscale('log')
plt.xscale('log')